In [15]:
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report
from tqdm import tqdm

In [30]:
MODEL_PATH = "/home/yeezy/100xDevs/personality/Personality-Classifier/models/double_chin_model.h5"

DATA_PATH = "/home/yeezy/100xDevs/personality/Personality-Classifier/data/processed/jaw_dataset/valid"


IMG_SIZE = (224, 224)

In [31]:
model = tf.keras.models.load_model(MODEL_PATH)
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 222, 222, 32)      896       
                                                                 
 max_pooling2d (MaxPooling2  (None, 111, 111, 32)      0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 109, 109, 64)      18496     
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 54, 54, 64)        0         
 g2D)                                                            
                                                                 
 conv2d_2 (Conv2D)           (None, 52, 52, 128)       73856     
                                                                 
 max_pooling2d_2 (MaxPoolin  (None, 26, 26, 128)       0

In [32]:
def prepare(image_path):
    img = cv2.imread(image_path)

    if img is None:
        return None

    img = cv2.resize(img, IMG_SIZE)
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    return img

In [34]:
y_true = []
y_probs = []

for label in ["double_chin", "no_double_chin"]:

    folder = "/home/yeezy/100xDevs/personality/Personality-Classifier/data/processed/jaw_dataset/valid/" + label

    for file in tqdm(os.listdir(folder)):

        img_path = os.path.join(folder, file)
        img = prepare(img_path)

        if img is None:
            continue

        prob = model.predict(img, verbose=0)[0][0]

        y_probs.append(prob)

        if label == "double_chin":
            y_true.append(1)
        else:
            y_true.append(0)

print("Total samples:", len(y_true))

 63%|██████▎   | 14321/22636 [39:04<32:16:59, 13.98s/it] 

: 

In [ ]:
plt.hist(
    [np.array(y_probs)[np.array(y_true)==1],
     np.array(y_probs)[np.array(y_true)==0]],
    bins=50,
    label=["Double Chin", "No Double Chin"]
)

plt.legend()
plt.title("Probability Distribution")
plt.xlabel("Predicted Probability")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.hist(
    [np.array(y_probs)[np.array(y_true)==1],
     np.array(y_probs)[np.array(y_true)==0]],
    bins=50,
    label=["Double Chin", "No Double Chin"]
)

plt.legend()
plt.title("Probability Distribution")
plt.xlabel("Predicted Probability")
plt.ylabel("Count")
plt.show()

🧠 2️⃣ How To Interpret ROC Curve
AUC Score Meaning
AUC	Meaning
0.5	Random
0.6–0.7	Weak
0.7–0.8	Acceptable
0.8–0.9	Strong
0.9+	Excellent

For personality system:
👉 You want at least 0.75+

In [ ]:
j_scores = tpr - fpr
best_index = np.argmax(j_scores)
best_threshold = thresholds[best_index]

print("Optimal Threshold:", best_threshold)

🟢 Cell 9 — Evaluate At Optimal Threshold

In [ ]:
y_pred = (np.array(y_probs) > best_threshold).astype(int)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))